In [1]:
# =================================================================================================
# Raw data : NHANES 2003-2014 Demographics, Laboratory, and Questionnaire datasets
# Purpose : Create one unified participant-level NHANES dataset by aggregating all domains using SEQN.
# Specification : Each NHANES dataset has SEQN "Participant ID", which is the unique ID used for merge.
# Files Used :
#   neighbourhood_nhanes_demographics_2003_2014_final.csv
#   neighbourhood_nhanes_labs_2003_2014_appended.csv
#   neighbourhood_nhanes_questionnaire_2003_2014_appended.csv
# Data Dictionary : NHANES 2003-2014 documentation
# Data Availability Matrix : Generated in this notebook using domain presence flags
# Source Site : https://wwwn.cdc.gov/nchs/nhanes/
# Developer Notes : Full outer join is used so no SEQN is dropped from any file.
# File Creator : Neha Khairnar
# =================================================================================================

# 📦 Step 1: Imports
import pandas as pd
from pathlib import Path

# =================================================================================================
# 📂 Step 2: Define Paths
# =================================================================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data" / "processed_v3" / "2009_2010"
demo_file = DATA_DIR / "neighbourhood_nhanes_demographics_2009_2010_raw_corrected.csv"
labs_file = DATA_DIR / "neighbourhood_nhanes_labs_2009_2010_raw_corrected.csv"
questionnaire_file = DATA_DIR / "neighbourhood_nhanes_questionnaire_2009_2010_raw_corrected.csv"
Examination_file = DATA_DIR / "neighbourhood_nhanes_examination_2009_2010_raw_corrected.csv"
Dietary_file = DATA_DIR / "neighbourhood_nhanes_dietary_2009_2010_raw_corrected.csv"


# =================================================================================================
# 📥 Step 3: Load Files
# =================================================================================================

demo = pd.read_csv(demo_file, dtype=str, keep_default_na=False)
labs = pd.read_csv(labs_file, dtype=str, keep_default_na=False)
questionnaire = pd.read_csv(questionnaire_file, dtype=str, keep_default_na=False)
Examination = pd.read_csv(Examination_file, dtype=str, keep_default_na=False)
Dietary = pd.read_csv(Dietary_file, dtype=str, keep_default_na=False)

print("📊 Loaded Files:")
print("Demographics:", demo.shape)
print("Labs:", labs.shape)
print("Questionnaire:", questionnaire.shape)
print("Examination:", Examination.shape)
print("Dietary:", Dietary.shape)

# =================================================================================================
# 🧹 Step 4: Standardize Column Names
# =================================================================================================
demo.columns = demo.columns.str.upper()
labs.columns = labs.columns.str.upper()
questionnaire.columns = questionnaire.columns.str.upper()
Examination.columns = Examination.columns.str.upper()
Dietary.columns = Dietary.columns.str.upper()

# =================================================================================================
# ✅ Step 5: Validate SEQN Exists
# =================================================================================================

for name, df_check in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire,
    "Examination":Examination,
    "Dietary":Dietary
}.items():
    if "SEQN" not in df_check.columns:
        raise ValueError(f"❌ SEQN column missing in {name} file")
    else:
        print(f"✅ SEQN found in {name}")

# =================================================================================================
# 📊 Step 6: File Structure Summary
# =================================================================================================
summary = []
for name, df_temp in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire,
    "Examination":Examination,
    "Dietary":Dietary
}.items():
    summary.append({
        "file": name,
        "rows": df_temp.shape[0],
        "columns": df_temp.shape[1],
        "unique_seqn": df_temp["SEQN"].nunique(),
        "duplicate_seqn_count": df_temp.duplicated(subset=["SEQN"]).sum(),
        "column_names": list(df_temp.columns)
    })

summary_df = pd.DataFrame(summary)
display(summary_df[["file", "rows", "columns", "unique_seqn", "duplicate_seqn_count"]])

final_df = (
    demo
    .merge(questionnaire, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_QUESTIONNAIRE"))
    .merge(labs, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_LAB"))
    .merge(Examination, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_EXAMINATION"))
    .merge(Dietary, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_DIETARY"))
)
# =================================================================================================
# 👀 Step 10: Validate Final Unified Dataset
# =================================================================================================

print("📊 Final Unified Dataset Shape:", final_df.shape)
print("Unique SEQN Count:", final_df["SEQN"].nunique())
print("Duplicate SEQN Count:", final_df.duplicated(subset=["SEQN"]).sum())
display(final_df.head(20))

OUTPUT_DIR = BASE_DIR / "data" / "processed_v3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_unified_2009_2010_corrected.csv"
final_df.to_csv(OUTPUT_PATH, index=False)
print("\n✅ Final unified NHANES dataset saved to:")
print(OUTPUT_PATH)


📊 Loaded Files:
Demographics: (10537, 45)
Labs: (9835, 35)
Questionnaire: (10537, 397)
Examination: (10253, 23)
Dietary: (10253, 244)
✅ SEQN found in demographics
✅ SEQN found in labs
✅ SEQN found in questionnaire
✅ SEQN found in Examination
✅ SEQN found in Dietary


,file,rows,columns,unique_seqn,duplicate_seqn_count
0,demographics,10537,45,10537,0
1,labs,9835,35,9835,0
2,questionnaire,10537,397,10537,0
3,Examination,10253,23,10253,0
4,Dietary,10253,244,10253,0


📊 Final Unified Dataset Shape: (10537, 740)
Unique SEQN Count: 10537
Duplicate SEQN Count: 0


,SEQN,SDDSRVYR,RIDSTATR,RIDEXMON,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDAGEEX,RIDRETH1,DMQMILIT,...,DR2TP184,DR2TP204,DR2TP205,DR2TP225,DR2TP226,DR2_300,DR2_320Z,DR2_330Z,DR2BWATZ,DR2TWS
0,51624,6,2,1,1,34,409,410,3,2,...,0.118,0.079,0.008,0.0,0.0,3,474.0,0.0,474.0,4
1,51625,6,2,2,1,4,49,50,5,,...,0.061,0.23199999999999998,0.004,0.015,0.045000000000000005,2,340.7,340.7,0.0,1
2,51626,6,2,1,1,16,202,202,4,,...,0.001,0.12100000000000001,0.008,0.041,0.003,2,0.0,0.0,0.0,1
3,51627,6,2,1,1,10,131,132,4,,...,0.0,0.031,0.004,0.004,0.001,3,78.99,78.99,0.0,1
4,51628,6,2,2,2,60,722,722,4,2,...,0.0,0.059,0.004,0.016,0.002,3,1036.88,1036.88,0.0,1
5,51629,6,2,1,1,26,313,313,1,2,...,0.004,0.5750000000000001,0.015000000000000001,0.083,0.087,3,1184.5,0.0,1184.5,4
6,51630,6,2,1,2,49,596,596,3,2,...,0.0,0.34600000000000003,0.03,0.061,0.122,2,500.0,0.0,500.0,4
7,51631,6,2,1,2,1,12,12,3,,...,,,,,,,,,,
8,51632,6,2,1,1,10,124,126,2,,...,,,,,,,,,,
9,51633,6,2,2,1,80,,,3,2,...,0.0,0.255,0.004,0.03,0.062000000000000006,2,1659.0,1659.0,0.0,1



✅ Final unified NHANES dataset saved to:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\processed_v3\neighbourhood_nhanes_unified_2009_2010_corrected.csv
